In [48]:
import pandas as pd
import numpy as np
import ast
import re

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
movies_metadata = pd.read_csv('movies_metadata.csv', low_memory=False)

movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
movies['clean_title'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()

def normalize_title(title):
    if isinstance(title, str):
        title = title.lower()  
        title = re.sub(r'[^a-z0-9\s]', '', title)  
        title = re.sub(r'\s+', ' ', title) 
        return title.strip()
    else:
        return ''

movies['normalized_title'] = movies['clean_title'].apply(normalize_title)

movies_metadata['release_date'] = pd.to_datetime(movies_metadata['release_date'], errors='coerce')
movies_metadata['year'] = movies_metadata['release_date'].dt.year
movies_metadata['clean_title'] = movies_metadata['title'].str.strip()
movies_metadata['normalized_title'] = movies_metadata['clean_title'].apply(normalize_title)

movies_metadata = movies_metadata.drop_duplicates(subset=['normalized_title', 'year'])

merged_movies = pd.merge(
    movies,
    movies_metadata[['normalized_title', 'year', 'production_companies']],
    on=['normalized_title', 'year'],
    how='left'
)

unmatched_movies = merged_movies[merged_movies['production_companies'].isnull()]

unmatched_movies = unmatched_movies.drop(['production_companies'], axis=1)

additional_merge = pd.merge(
    unmatched_movies,
    movies_metadata[['normalized_title', 'production_companies']],
    on='normalized_title',
    how='left'
)

matched_movies = merged_movies[merged_movies['production_companies'].notnull()]
merged_movies = pd.concat([matched_movies, additional_merge], ignore_index=True)

def get_all_production_companies(x):
    try:
        companies = ast.literal_eval(x)
        return [company['name'] for company in companies] if companies else []
    except (ValueError, SyntaxError, TypeError):
        return []

merged_movies['production_companies_list'] = merged_movies['production_companies'].apply(get_all_production_companies)

movies_exploded = merged_movies.explode('production_companies_list')

movies_exploded = movies_exploded.dropna(subset=['production_companies_list'])

most_common_company = movies_exploded.groupby('movieId')['production_companies_list'] \
    .apply(lambda x: x.value_counts().idxmax())

merged_movies = merged_movies.merge(
    most_common_company.rename('most_common_production_company'),
    on='movieId',
    how='left'
)

merged_movies['provider'] = merged_movies['most_common_production_company'].fillna('Unknown')

item_provider_mapping = merged_movies[['movieId', 'title', 'provider']].drop_duplicates()

ratings_with_providers = ratings.merge(item_provider_mapping, on='movieId', how='left')

ratings_with_providers['provider'] = ratings_with_providers['provider'].fillna('Unknown')

ratings_with_providers.to_csv('ratings_with_providers.csv', index=False)


In [50]:
total_movies = merged_movies['movieId'].nunique()
known_providers = merged_movies[merged_movies['provider'] != 'Unknown']['movieId'].nunique()
unknown_providers = total_movies - known_providers

print(f"Total movies: {total_movies}")
print(f"Movies with known providers: {known_providers}")
print(f"Movies with 'Unknown' providers: {unknown_providers}")
print(f"Percentage with known providers: {known_providers / total_movies * 100:.2f}%")

Total movies: 9742
Movies with known providers: 6232
Movies with 'Unknown' providers: 3510
Percentage with known providers: 63.97%
